In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

data_dir = r"C:\Users\patel\Downloads\archive_1\PlantVillage"
IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),   # slightly wider crop range than v1
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),                               # a bit more than v1's 20
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.35, hue=0.05),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.3),   # NEW — simulates off-angle field photos
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))], p=0.3),  # NEW — simulates camera blur
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(f"{data_dir}\\train", transform=train_transform)
val_dataset   = datasets.ImageFolder(f"{data_dir}\\val", transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

num_classes = len(train_dataset.classes)
print(f"Classes: {num_classes} | Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Classes: 38 | Train: 43444 | Val: 10861


In [2]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)  # bigger backbone than v1's resnet18
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

print(f"Model ready (ResNet50). Will output {num_classes}-way predictions.")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\patel/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


  0%|          | 0.00/97.8M [00:00<?, ?B/s]

  0%|          | 384k/97.8M [00:00<00:27, 3.76MB/s]

  1%|          | 896k/97.8M [00:00<00:27, 3.64MB/s]

  1%|▏         | 1.25M/97.8M [00:00<00:29, 3.47MB/s]

...

 99%|█████████▊| 96.5M/97.8M [00:27<00:00, 3.52MB/s]

 99%|█████████▉| 97.0M/97.8M [00:27<00:00, 3.24MB/s]

100%|█████████▉| 97.5M/97.8M [00:27<00:00, 3.20MB/s]

100%|██████████| 97.8M/97.8M [00:27<00:00, 3.68MB/s]

Model ready (ResNet50). Will output 38-way predictions.


In [3]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)   # NEW vs v1 — discourages overconfidence
optimizer = optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 10
print("Ready.")

Ready.


In [4]:
import time
from sklearn.metrics import f1_score

best_macro_f1 = -1.0
best_model_state = None

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    start_time = time.time()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_dataset)

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    val_macro_f1 = f1_score(all_labels, all_preds, average="macro")
    elapsed = time.time() - start_time

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} | "
          f"val_macro_f1={val_macro_f1:.4f} | time={elapsed:.1f}s")

    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  ↳ New best! Saved checkpoint (macro-F1={best_macro_f1:.4f})")

print(f"\nTraining done. Best validation macro-F1: {best_macro_f1:.4f}")

Epoch 1/10 | train_loss=0.9798 | val_macro_f1=0.9844 | time=188.9s
  ↳ New best! Saved checkpoint (macro-F1=0.9844)


Epoch 2/10 | train_loss=0.7445 | val_macro_f1=0.9915 | time=188.8s
  ↳ New best! Saved checkpoint (macro-F1=0.9915)


Epoch 3/10 | train_loss=0.7232 | val_macro_f1=0.9903 | time=188.8s


Epoch 4/10 | train_loss=0.7148 | val_macro_f1=0.9907 | time=188.8s


Epoch 5/10 | train_loss=0.7078 | val_macro_f1=0.9920 | time=188.3s
  ↳ New best! Saved checkpoint (macro-F1=0.9920)


Epoch 6/10 | train_loss=0.7049 | val_macro_f1=0.9925 | time=186.6s
  ↳ New best! Saved checkpoint (macro-F1=0.9925)


Epoch 7/10 | train_loss=0.7009 | val_macro_f1=0.9895 | time=190.5s


Epoch 8/10 | train_loss=0.7000 | val_macro_f1=0.9907 | time=188.7s


Epoch 9/10 | train_loss=0.6982 | val_macro_f1=0.9946 | time=189.5s
  ↳ New best! Saved checkpoint (macro-F1=0.9946)


Epoch 10/10 | train_loss=0.6948 | val_macro_f1=0.9906 | time=189.2s

Training done. Best validation macro-F1: 0.9946


In [5]:
from sklearn.metrics import confusion_matrix, classification_report

model.load_state_dict(best_model_state)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

cm = confusion_matrix(all_labels, all_preds)
report = classification_report(all_labels, all_preds, target_names=train_dataset.classes, digits=3)

print("=== Per-class report (v2) ===")
print(report)

=== Per-class report (v2) ===
                                                    precision    recall  f1-score   support

                                Apple___Apple_scab      1.000     1.000     1.000       126
                                 Apple___Black_rot      1.000     1.000     1.000       125
                          Apple___Cedar_apple_rust      1.000     1.000     1.000        55
                                   Apple___healthy      1.000     1.000     1.000       329
                               Blueberry___healthy      1.000     1.000     1.000       300
          Cherry_(including_sour)___Powdery_mildew      1.000     1.000     1.000       210
                 Cherry_(including_sour)___healthy      1.000     0.988     0.994       170
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot      0.907     0.942     0.924       103
                       Corn_(maize)___Common_rust_      1.000     1.000     1.000       239
               Corn_(maize)___Northern_Leaf_Bligh

In [6]:
import pickle
import os

bundle = {
    "architecture": "resnet50",           # NOTE: changed from resnet18 — predict.py must match this
    "num_classes": num_classes,
    "class_names": train_dataset.classes,
    "state_dict": model.state_dict(),
    "img_size": IMG_SIZE,
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std": [0.229, 0.224, 0.225],
}

with open("model_v2.pkl", "wb") as f:
    pickle.dump(bundle, f)

size_mb = os.path.getsize("model_v2.pkl") / (1024 * 1024)
print(f"Saved model_v2.pkl ({size_mb:.1f} MB)")

Saved model_v2.pkl (90.3 MB)


In [7]:
import datetime

overall_accuracy = (torch.tensor(all_preds) == torch.tensor(all_labels)).float().mean().item()

print("=" * 70)
print("AGRISMART AI — MODEL SUMMARY (v2)")
print("=" * 70)
print(f"\n[MODEL] ResNet50 | label_smoothing=0.1 | stronger augmentation")
print(f"[DATASET] Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"\n[HEADLINE METRICS]")
print(f"  Macro-F1: {best_macro_f1:.4f}")
print(f"  Accuracy: {overall_accuracy:.4f}")

from sklearn.metrics import f1_score as f1_per_class
per_class_f1 = f1_per_class(all_labels, all_preds, average=None)
class_f1_pairs = sorted(zip(train_dataset.classes, per_class_f1), key=lambda x: x[1])

print(f"\n[WEAKEST 5 CLASSES]")
for name, f1 in class_f1_pairs[:5]:
    print(f"  {name}: F1={f1:.3f}")

print(f"\n[GENERATED] {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 70)

summary_text = f"""AgriSmart AI — Model Summary Report (v2)
Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

MODEL: ResNet50, label_smoothing=0.1, stronger augmentation (perspective + blur)
DATASET: Train={len(train_dataset)}, Val={len(val_dataset)}

HEADLINE METRICS
  Macro-F1: {best_macro_f1:.4f}
  Accuracy: {overall_accuracy:.4f}

WEAKEST CLASSES
{chr(10).join(f"  {name}: F1={f1:.3f}" for name, f1 in class_f1_pairs[:5])}

FULL PER-CLASS REPORT
{report}
"""

with open("model_summary_v2.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\nSaved model_summary_v2.txt")

AGRISMART AI — MODEL SUMMARY (v2)

[MODEL] ResNet50 | label_smoothing=0.1 | stronger augmentation
[DATASET] Train: 43444 | Val: 10861

[HEADLINE METRICS]
  Macro-F1: 0.9946
  Accuracy: 0.9967

[WEAKEST 5 CLASSES]
  Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: F1=0.924
  Corn_(maize)___Northern_Leaf_Blight: F1=0.959
  Potato___healthy: F1=0.984
  Tomato___Target_Spot: F1=0.984
  Tomato___Early_blight: F1=0.985

[GENERATED] 2026-09-11 18:27:17

Saved model_summary_v2.txt
